<a href="https://www.kaggle.com/code/avikdas567/digitization-of-ecg-images?scriptVersionId=291563032" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports & Config

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Load Test Metadata

In [2]:
TEST_CSV = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_SUB = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"

test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_parquet(SAMPLE_SUB)

print("Test rows:", len(test_df))
print("Submission rows:", len(sample_sub))

Test rows: 24
Submission rows: 75000


# ECG Length Helper

In [3]:
def expected_length(fs, lead):
    if lead == "II":
        return int(np.floor(fs * 10.0))
    else:
        return int(np.floor(fs * 2.5))

# Baseline ECG Generator

In [4]:
def generate_baseline_ecg(length, fs):
    t = np.arange(length) / fs
    
    signal = 0.01 * np.sin(2 * np.pi * 1.2 * t)
    
    noise = 0.002 * np.random.randn(length)
    
    return signal + noise

# Generate Predictions

In [5]:
records = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    base_id = row["id"]
    fs = row["fs"]
    lead = row["lead"]
    
    length = expected_length(fs, lead)
    signal = generate_baseline_ecg(length, fs)
    
    for i, val in enumerate(signal):
        records.append({
            "id": f"{base_id}_{i}_{lead}",
            "value": float(val)
        })

100%|██████████| 24/24 [00:00<00:00, 155.84it/s]


# Create Submission File

In [6]:
submission = pd.DataFrame(records)

submission = submission.set_index("id").loc[sample_sub["id"]].reset_index()

submission.head()

,id,value
0,1053922973_0_I,0.001620
1,1053922973_1_I,-0.001351
2,1053922973_2_I,0.002416
3,1053922973_3_I,-0.000477
4,1053922973_4_I,-0.001785


# Save Submission

In [7]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Rows:", len(submission))

Saved: submission.csv
Rows: 75000
